<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-ComputerVision/blob/main/Klassische_Segmentierung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Klassische Segmentierungsverfahren

**Computer Vision – BSc Business Artificial Intelligence – FHNW**

In diesem Notebook lernst du vier klassische Verfahren der Bildsegmentierung kennen, *bevor* wir uns in den nächsten Sessions mit Deep-Learning-basierten Methoden beschäftigen. Das hilft dir zu verstehen, *warum* die heutige Forschung auf neuronale Netze setzt – und wann ein einfaches klassisches Verfahren völlig ausreicht.

## Was du hier machst

| Verfahren | Idee in einem Satz |
|---|---|
| **Thresholding (Otsu)** | Pixel anhand ihrer Helligkeit in zwei Klassen aufteilen |
| **Region Growing** | Von einem Saatpunkt aus Pixel "anbauen", die ähnlich sind |
| **Watershed** | Das Bild wie eine Landschaft fluten und an den "Wasserscheiden" trennen |
| **GrabCut** | Mit einer Box um das Objekt herum den Vordergrund herausschneiden |

## Lernziele

Nach diesem Notebook kannst du:
- die vier Verfahren selbst auf eigene Bilder anwenden
- erklären, *wann* ein Verfahren funktioniert und wann nicht
- sinnvoll einschätzen, ob du für ein Problem überhaupt Deep Learning brauchst

> 💡 **Hinweis zur Nutzung:** Im Notebook gibt es Stellen mit `???`. Dort sollst du selbst etwas eintragen. Die Lösungen findest du jeweils in einem aufklappbaren Block direkt darunter – aber probier es bitte **zuerst selbst**!

## 1. Setup

Wir importieren die nötigen Bibliotheken. Auf Google Colab sind diese alle bereits installiert.

In [ ]:
# Falls eine Bibliothek fehlt, hier installieren (auf Colab i.d.R. nicht nötig)
# !pip install -q scikit-image opencv-python pooch

# Standard-Imports
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import data, filters, segmentation, color, morphology, measure
from skimage.segmentation import flood, watershed
from scipy import ndimage as ndi

# Plot-Einstellungen
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['image.cmap'] = 'gray'

# Umgebung erkennen (Colab vs. lokal)
try:
    from google.colab import files
    IN_COLAB = True
    print("Du arbeitest in Google Colab.")
except ImportError:
    IN_COLAB = False
    print("Du arbeitest in einer lokalen Jupyter-Umgebung.")

## 2. Beispielbilder laden

Wir nutzen Beispielbilder aus `scikit-image`. Sie sind klein, schnell und decken verschiedene Schwierigkeitsgrade ab.

In [ ]:
# Drei sehr unterschiedliche Bilder
coins = data.coins()              # Graustufen, Münzen auf einfachem Hintergrund
astronaut = data.astronaut()      # RGB, Person vor Hintergrund (für GrabCut)
page_img = data.page()            # Dokumentenscan (klassisches Thresholding-Beispiel)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(coins);     axes[0].set_title('coins (einfach)');             axes[0].axis('off')
axes[1].imshow(astronaut); axes[1].set_title('astronaut (mittel)');          axes[1].axis('off')
axes[2].imshow(page_img);  axes[2].set_title('page (ungleichm. Beleuchtung)'); axes[2].axis('off')
plt.tight_layout()
plt.show()

print(f"coins:     shape={coins.shape},     dtype={coins.dtype}")
print(f"astronaut: shape={astronaut.shape}, dtype={astronaut.dtype}")
print(f"page:      shape={page_img.shape},  dtype={page_img.dtype}")

### Eigenes Bild hochladen (optional, aber empfohlen)

Für die spätere Reflexion ist es wertvoll, wenn du auch ein **eigenes Bild** ausprobierst. Lade dazu unten ein Foto hoch (am besten etwas mit einem klaren Vordergrund-Objekt, z. B. ein Kaffeebecher auf dem Tisch, eine Pflanze, etc.).

In [ ]:
# Eigenes Bild hochladen
my_image = None  # bleibt None, falls du nichts hochlädst

if IN_COLAB:
    print("Wähle ein Bild aus (oder klicke 'Cancel', um diesen Schritt zu überspringen):")
    try:
        uploaded = files.upload()
        if uploaded:
            filename = list(uploaded.keys())[0]
            my_image = cv2.cvtColor(cv2.imread(filename), cv2.COLOR_BGR2RGB)
            print(f"Geladen: {filename}, shape={my_image.shape}")
    except Exception as e:
        print(f"Kein Bild hochgeladen ({e}).")
else:
    # Lokal: Pfad anpassen, falls gewünscht
    # my_image = cv2.cvtColor(cv2.imread('mein_bild.jpg'), cv2.COLOR_BGR2RGB)
    print("Lokale Umgebung: Pfad im Code anpassen, falls du ein eigenes Bild verwenden möchtest.")

if my_image is not None:
    plt.figure(figsize=(8, 6))
    plt.imshow(my_image); plt.axis('off'); plt.title('Dein Bild')
    plt.show()

---
## 3. Methode 1: Thresholding (Schwellwertverfahren)

### Intuition

Stell dir das Histogramm eines Bildes vor: bei einem Bild mit hellen Objekten auf dunklem Hintergrund (oder umgekehrt) hat das Histogramm zwei "Berge" (bimodale Verteilung). Thresholding sucht das **Tal** dazwischen und sagt: *"Alles links davon ist Hintergrund, alles rechts davon ist Vordergrund."*

**Otsus Methode** macht das automatisch: Sie findet den Schwellwert, der die *Varianz innerhalb* der beiden Klassen minimiert (= die *Varianz zwischen* den Klassen maximiert).

$$\text{threshold}^* = \arg\min_t \; \sigma_{\text{innerhalb}}^2(t)$$

Du musst die Formel nicht auswendig kennen – wichtig ist die Idee: *automatisch das beste Tal finden*.

### 3.1 Histogramm anschauen

Bevor wir einen Schwellwert wählen, schauen wir uns das Histogramm an. Erkennst du die zwei Berge?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(coins); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].hist(coins.ravel(), bins=256, color='steelblue')
axes[1].set_title('Histogramm der Pixel-Helligkeiten')
axes[1].set_xlabel('Pixelwert (0 = schwarz, 255 = weiss)')
axes[1].set_ylabel('Anzahl Pixel')
plt.tight_layout(); plt.show()

### 3.2 Manueller Schwellwert

Probiere verschiedene Werte aus. Was passiert bei sehr tiefen / sehr hohen Werten?

In [ ]:
# AUFGABE: Probiere mindestens 3 verschiedene Schwellwerte aus.
# Welcher liefert die beste Trennung von Münzen und Hintergrund?

threshold_value = ???   # z.B. 100, 130, 160, ...

binary = coins > threshold_value

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(coins);  axes[0].set_title('Original');                       axes[0].axis('off')
axes[1].imshow(binary); axes[1].set_title(f'Binär (Schwelle = {threshold_value})'); axes[1].axis('off')
plt.tight_layout(); plt.show()

print(f"Anteil Vordergrund-Pixel: {binary.mean():.2%}")

<details>
<summary>💡 Lösung anzeigen</summary>

Werte um **120–130** liefern eine brauchbare Trennung. Bei zu tiefen Werten (< 80) wird zu viel als Vordergrund klassifiziert, bei zu hohen (> 180) verschwinden Teile der Münzen.

</details>

### 3.3 Otsus Methode: automatischer Schwellwert

In [ ]:
# Otsu findet den Schwellwert automatisch
otsu_threshold = filters.threshold_otsu(coins)
binary_otsu = coins > otsu_threshold

print(f"Otsu hat den Schwellwert {otsu_threshold} gewählt.")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(coins);       axes[0].set_title('Original');                       axes[0].axis('off')
axes[1].hist(coins.ravel(), bins=256, color='steelblue')
axes[1].axvline(otsu_threshold, color='red', linestyle='--', label=f'Otsu = {otsu_threshold}')
axes[1].set_title('Histogramm + Otsu-Schwelle'); axes[1].legend()
axes[2].imshow(binary_otsu); axes[2].set_title(f'Otsu-Resultat'); axes[2].axis('off')
plt.tight_layout(); plt.show()

### 3.4 Wenn Thresholding versagt

Otsu funktioniert gut bei bimodalen Histogrammen. Was passiert bei einem komplexeren Bild?

In [ ]:
# Astronaut in Graustufen umwandeln
astronaut_gray = color.rgb2gray(astronaut)
astronaut_gray_uint = (astronaut_gray * 255).astype(np.uint8)

otsu_astro = filters.threshold_otsu(astronaut_gray_uint)
binary_astro = astronaut_gray_uint > otsu_astro

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(astronaut);                          axes[0].set_title('Original');             axes[0].axis('off')
axes[1].hist(astronaut_gray_uint.ravel(), bins=256, color='steelblue')
axes[1].axvline(otsu_astro, color='red', linestyle='--')
axes[1].set_title(f'Histogramm + Otsu-Schwelle ({otsu_astro})')
axes[2].imshow(binary_astro);                       axes[2].set_title('Otsu-Resultat');        axes[2].axis('off')
plt.tight_layout(); plt.show()

### 🎯 Reflexionsfrage

**Schau dir das Otsu-Resultat des Astronauten-Bildes an. Was ist hier das Problem?**

Notiere deine Beobachtung in der nächsten Zelle.

<details>
<summary>💡 Mögliche Antwort</summary>

Das Histogramm ist **nicht bimodal** – es gibt keine klaren zwei Berge, sondern eine breite Verteilung. Der Astronaut, der Helm und der Hintergrund haben überlappende Helligkeiten. Otsu trennt zwar, aber die Trennung ist semantisch unsinnig: Teile des Astronauten landen im "Hintergrund" und umgekehrt.

**Take-away:** Thresholding funktioniert nur, wenn Helligkeit alleine ausreicht, um Objekt und Hintergrund zu trennen.

</details>

---
## 4. Methode 2: Region Growing

### Intuition

Stell dir vor, du wirfst einen Tropfen Tinte auf ein Blatt Papier. Die Tinte breitet sich aus – aber nur dort, wo das Papier ähnlich ist (z. B. nicht über eine Wachsschicht hinaus).

Bei Region Growing machen wir genau das:
1. Wähle einen **Saatpunkt** (Seed) im Objekt.
2. Schau dir alle Nachbarn an.
3. Falls ein Nachbar dem Saatpunkt **ähnlich** ist (Helligkeit, Farbe, ...), nimm ihn dazu.
4. Wiederhole, bis nichts mehr wächst.

In `scikit-image` gibt es dafür die Funktion `flood()`.

In [ ]:
# Saatpunkt setzen (Zeile, Spalte) – ungefähr in einer der Münzen
seed_point = (120, 100)

# Toleranz: wie ähnlich muss ein Nachbar sein?
tolerance = 30

mask = flood(coins, seed_point, tolerance=tolerance)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(coins)
axes[0].plot(seed_point[1], seed_point[0], 'r*', markersize=20)
axes[0].set_title('Original mit Saatpunkt'); axes[0].axis('off')

axes[1].imshow(mask); axes[1].set_title(f'Gewachsene Region (Toleranz={tolerance})'); axes[1].axis('off')

# Overlay
overlay = coins.copy()
overlay_rgb = np.stack([overlay]*3, axis=-1).astype(np.uint8)
overlay_rgb[mask] = [255, 0, 0]
# leichtes Blending
blended = (0.6 * np.stack([coins]*3, axis=-1).astype(np.uint8) + 0.4 * overlay_rgb).astype(np.uint8)
axes[2].imshow(blended); axes[2].set_title('Region als Overlay'); axes[2].axis('off')
plt.tight_layout(); plt.show()

### 4.1 Eigene Experimente

Probiere folgende Variationen:
- Setze den Saatpunkt **in den Hintergrund** (z. B. `(20, 20)`).
- Variiere die Toleranz (`5`, `15`, `30`, `60`).

**Was beobachtest du?**

In [ ]:
# AUFGABE: Erstelle ein 2x2-Plot mit 4 verschiedenen Toleranzen.
# Verwende denselben Saatpunkt für alle vier.

seed_point = (100, 100)
tolerances = [???, ???, ???, ???]   # z.B. [5, 15, 30, 60]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, tol in zip(axes.ravel(), tolerances):
    mask = flood(coins, seed_point, tolerance=tol)
    ax.imshow(coins)
    ax.contour(mask, colors='red', linewidths=2)
    ax.plot(seed_point[1], seed_point[0], 'y*', markersize=15)
    ax.set_title(f'Toleranz = {tol}')
    ax.axis('off')
plt.tight_layout(); plt.show()

### ✅ Checkpoint

Notiere unten **zwei konkrete Beobachtungen aus deinen Experimenten**, basierend auf den von dir gewählten Toleranzwerten.

In [ ]:
# Trage deine konkreten Werte und Beobachtungen ein:
beobachtung_region_growing = f'''
Ich habe folgende Toleranzen ausprobiert: {tolerances}

Beobachtung 1 (z.B. bei tiefer Toleranz): ???

Beobachtung 2 (z.B. bei hoher Toleranz): ???
'''
print(beobachtung_region_growing)

<details>
<summary>💡 Mögliche Beobachtungen</summary>

- Bei **kleiner Toleranz** (z. B. 5) wächst die Region kaum – nur wenige Pixel rund um den Saatpunkt werden eingeschlossen.
- Bei **grosser Toleranz** (z. B. 60+) "läuft" die Region oft *aus der Münze heraus* in den Hintergrund. Man spricht von **leakage**.
- Ein gut gewählter Mittelwert (15–30) erfasst genau eine Münze – aber **nur diese eine**, nicht alle.

**Take-away:** Region Growing ist sehr empfindlich gegenüber dem Saatpunkt und der Toleranz. Es eignet sich gut für **interaktive Anwendungen** (Mensch wählt Saatpunkt) und schlecht für vollautomatische Pipelines.

</details>

---
## 5. Methode 3: Watershed (Wasserscheide)

### Intuition

Stell dir das Graustufenbild als **Landschaft** vor: helle Pixel sind Berge, dunkle Pixel sind Täler. Jetzt:
1. Stich an den lokalen Tiefpunkten ein Loch hinein.
2. Lass langsam Wasser aus jedem Loch aufsteigen.
3. Dort, wo zwei Wasserbecken aufeinandertreffen, wird ein **Damm** gebaut – das ist die Wasserscheide (engl. *watershed*).

Diese Dämme sind unsere Segmentgrenzen. Watershed eignet sich besonders für **sich berührende Objekte**, die Thresholding allein nicht trennen kann.

**Klassisches Beispiel:** Münzen, die sich berühren – Otsu macht daraus *einen* Klumpen, Watershed trennt sie sauber.

In [ ]:
# Schritt 1: Binärmaske mit Otsu (was ist Münze, was ist Hintergrund?)
threshold = filters.threshold_otsu(coins)
binary = coins > threshold

# Aufräumen: kleine Löcher schliessen
binary_clean = ndi.binary_fill_holes(binary)
binary_clean = morphology.remove_small_objects(binary_clean, min_size=100)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(binary);       axes[0].set_title('Otsu (roh)');     axes[0].axis('off')
axes[1].imshow(binary_clean); axes[1].set_title('Nach Aufräumen'); axes[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Schritt 2: Distanztransformation
# Für jeden Vordergrund-Pixel: wie weit ist der nächste Hintergrund-Pixel?
# -> Münzenzentren bekommen hohe Werte, Ränder niedrige
distance = ndi.distance_transform_edt(binary_clean)

# Lokale Maxima finden = Münzenzentren = unsere "Quellen"
from skimage.feature import peak_local_max
coords = peak_local_max(distance, min_distance=20, labels=binary_clean)
markers = np.zeros(distance.shape, dtype=int)
markers[tuple(coords.T)] = np.arange(1, len(coords) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(distance, cmap='magma'); axes[0].set_title('Distanztransformation'); axes[0].axis('off')
axes[1].imshow(coins)
axes[1].plot(coords[:, 1], coords[:, 0], 'r*', markersize=12)
axes[1].set_title(f'{len(coords)} gefundene Maxima (Münzenzentren)'); axes[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Schritt 3: Watershed anwenden
labels = watershed(-distance, markers, mask=binary_clean)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(coins);                           axes[0].set_title('Original');               axes[0].axis('off')
axes[1].imshow(color.label2rgb(labels, image=coins, bg_label=0, alpha=0.4))
axes[1].set_title(f'Watershed: {labels.max()} Segmente'); axes[1].axis('off')
plt.tight_layout(); plt.show()

print(f"Anzahl gefundener Münzen: {labels.max()}")

### ✅ Checkpoint

**Wie viele Münzen hat dein Watershed gefunden?** Trage die Zahl unten ein und zähle dann manuell auf dem Originalbild nach.

In [ ]:
anzahl_gefunden_watershed = ???   # was hat labels.max() ausgegeben?
anzahl_tatsaechlich     = ???   # zähl auf dem Bild nach!

print(f"Gefunden:    {anzahl_gefunden_watershed}")
print(f"Tatsächlich: {anzahl_tatsaechlich}")
print(f"Differenz:   {abs(anzahl_gefunden_watershed - anzahl_tatsaechlich)}")

### 5.1 Parameter-Spielwiese

Der Parameter `min_distance` in `peak_local_max` steuert, wie nah zwei Münzenzentren beieinander liegen dürfen. Probiere:

In [ ]:
# AUFGABE: Was passiert mit min_distance = 5? Mit min_distance = 50?
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, min_dist in zip(axes, [???, ???, ???]):   # z.B. [5, 20, 50]
    coords = peak_local_max(distance, min_distance=min_dist, labels=binary_clean)
    markers = np.zeros(distance.shape, dtype=int)
    markers[tuple(coords.T)] = np.arange(1, len(coords) + 1)
    labels = watershed(-distance, markers, mask=binary_clean)
    ax.imshow(color.label2rgb(labels, image=coins, bg_label=0, alpha=0.4))
    ax.set_title(f'min_distance = {min_dist} → {labels.max()} Segmente')
    ax.axis('off')
plt.tight_layout(); plt.show()

<details>
<summary>💡 Erwartetes Verhalten</summary>

- **min_distance = 5**: Zu viele Maxima → eine einzelne Münze wird in mehrere Segmente zerlegt (Übersegmentierung).
- **min_distance = 20**: Brauchbares Resultat, ungefähr eine Markierung pro Münze.
- **min_distance = 50**: Zu wenige Maxima → mehrere benachbarte Münzen verschmelzen zu einem Segment (Untersegmentierung).

**Take-away:** Watershed ist mächtig für sich berührende, ähnlich grosse Objekte – aber die Wahl der Marker ist entscheidend.

</details>

---
## 6. Methode 4: GrabCut

### Intuition

GrabCut ist *halb-automatisch*: Du sagst dem Algorithmus grob, *wo* das Objekt ist – z. B. mit einem Rechteck darum –, und der Algorithmus findet die genaue Kontur selbst.

Im Hintergrund modelliert GrabCut die Farbverteilungen von Vordergrund und Hintergrund (Gaussian Mixture Models) und löst ein **Graph-Cut-Problem**, das eine optimale Trennung findet. Die Details musst du dir nicht merken – wichtig ist:

> GrabCut = "Ich gebe einen groben Hinweis, der Algorithmus macht den Rest."

Das ist genau die Art interaktiver Werkzeug, die du heute in vielen Foto-Apps unter Namen wie *"Magic Select"* oder *"Hintergrund entfernen"* findest – wobei moderne Apps inzwischen Deep Learning verwenden.

In [ ]:
# OpenCV erwartet BGR – also umwandeln
img_bgr = cv2.cvtColor(astronaut, cv2.COLOR_RGB2BGR)

# Bounding Box: (x, y, breite, höhe) – grob um die Astronautin herum
rect = (50, 50, 400, 460)

# Initialisierung
mask = np.zeros(img_bgr.shape[:2], np.uint8)
bgd_model = np.zeros((1, 65), np.float64)
fgd_model = np.zeros((1, 65), np.float64)

# GrabCut ausführen (5 Iterationen)
cv2.grabCut(img_bgr, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)

# Maske aufbereiten: Werte 0 und 2 = Hintergrund, 1 und 3 = Vordergrund
final_mask = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')
result = astronaut * final_mask[:, :, np.newaxis]

# Visualisierung
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(astronaut)
from matplotlib.patches import Rectangle
axes[0].add_patch(Rectangle((rect[0], rect[1]), rect[2], rect[3],
                             fill=False, edgecolor='red', linewidth=2))
axes[0].set_title('Original mit Bounding Box'); axes[0].axis('off')
axes[1].imshow(final_mask); axes[1].set_title('GrabCut-Maske'); axes[1].axis('off')
axes[2].imshow(result);     axes[2].set_title('Ergebnis');      axes[2].axis('off')
plt.tight_layout(); plt.show()

### 6.1 Eigene Anwendung

Probiere GrabCut an deinem **eigenen Bild** aus (das du oben hochgeladen hast). Setze die Bounding Box so, dass sie grob das Objekt umschliesst, das du segmentieren willst.

In [ ]:
if my_image is not None:
    img_bgr_own = cv2.cvtColor(my_image, cv2.COLOR_RGB2BGR)

    h, w = my_image.shape[:2]
    print(f"Dein Bild ist {w} breit und {h} hoch.")

    # AUFGABE: Setze die Bounding Box passend zu deinem Bild!
    # Format: (x_links_oben, y_links_oben, breite, höhe)
    rect_own = (???, ???, ???, ???)

    mask = np.zeros(img_bgr_own.shape[:2], np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    cv2.grabCut(img_bgr_own, mask, rect_own, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
    final_mask_own = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')
    result_own = my_image * final_mask_own[:, :, np.newaxis]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(my_image)
    axes[0].add_patch(Rectangle((rect_own[0], rect_own[1]), rect_own[2], rect_own[3],
                                  fill=False, edgecolor='red', linewidth=2))
    axes[0].set_title('Dein Bild + Box'); axes[0].axis('off')
    axes[1].imshow(final_mask_own); axes[1].set_title('Maske'); axes[1].axis('off')
    axes[2].imshow(result_own);     axes[2].set_title('Ergebnis');     axes[2].axis('off')
    plt.tight_layout(); plt.show()
else:
    print("Du hast kein eigenes Bild hochgeladen. Lade oben eines hoch und führe diese Zelle erneut aus.")

### ✅ Checkpoint

Beschreibe in der nächsten Zelle ehrlich:
1. Welches Objekt hast du segmentiert?
2. Wie gut war das Resultat (auf einer Skala von 1 = unbrauchbar bis 5 = perfekt)?
3. Wo macht GrabCut bei deinem Bild Fehler?

In [ ]:
beobachtung_grabcut = '''
Objekt: ???

Qualität (1-5): ???

Fehlerstellen: ???
'''
print(beobachtung_grabcut)

---
## 7. Zusammenfassung & Vergleich

### Wann eignet sich welches Verfahren?

| Verfahren | Stärken | Schwächen | Typischer Use-Case |
|---|---|---|---|
| **Thresholding** | Sehr schnell, sehr einfach | Nur bei klar unterschiedlichen Helligkeiten | Dokumentenscans, einfache Industriekontrolle |
| **Region Growing** | Lokale Kontrolle, interaktiv | Sehr seed-abhängig, leakage | Medizinische Bildgebung (Radiologie), interaktive Tools |
| **Watershed** | Trennt sich berührende Objekte | Tendiert zu Übersegmentierung | Zellzählung, Münzen-/Partikelzählung |
| **GrabCut** | Gute Qualität bei wenig Input | Braucht Benutzerinteraktion | Foto-Editoren, Hintergrund-Removal |

### 🎯 Abschlussreflexion

Beantworte für dich (gerne auch schriftlich in der nächsten Zelle):

1. Bei welchem deiner Bilder hat **welches** Verfahren am besten funktioniert?
2. Was haben **alle vier** Verfahren gemeinsam, was Deep Learning anders macht?
3. Wo siehst du in einem **Business-Kontext** (z. B. deinem Praktikumsfeld) eine Anwendung, bei der ein klassisches Verfahren völlig genügt – ganz ohne CNN?

In [ ]:
abschlussreflexion = '''
1. Bestes Verfahren bei meinen Bildern: ???

2. Gemeinsamkeit der klassischen Methoden vs. Deep Learning: ???

3. Business-Anwendung mit klassischem Verfahren: ???
'''
print(abschlussreflexion)